In [1]:
# Gerekli kütüphaneler
import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter
import ollama
import os
from rapidfuzz import fuzz

print("✅ Tüm kütüphaneler yüklendi!")
print("🇹🇷 Türkiye Bilgi Chatbot'u hazırlanıyor...")

c:\Users\MONSTER\Desktop\turkiye_chatbot\venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Tüm kütüphaneler yüklendi!
🇹🇷 Türkiye Bilgi Chatbot'u hazırlanıyor...


 Hücre 3: Tüm Dökümanları Okuyalım

In [2]:
import glob

def load_all_documents():
    """
    data/ klasöründeki tüm dökümanları oku
    """
    print("📚 Dökümanlar yükleniyor...\n")
    
    all_documents = []
    file_paths = glob.glob("data/*.txt")
    
    for file_path in file_paths:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
            all_documents.append({
                'content': content,
                'source': file_path.split('/')[-1].replace('.txt', '')
            })
        
        print(f"✅ {file_path} okundu ({len(content)} karakter)")
    
    # Tüm metni birleştir
    full_text = "\n\n".join([doc['content'] for doc in all_documents])
    
    print(f"\n📊 Toplam: {len(full_text)} karakter")
    print(f"📊 Toplam: {len(full_text.split())} kelime")
    
    return full_text, all_documents

# Dökümanları yükle
full_knowledge, documents = load_all_documents()

📚 Dökümanlar yükleniyor...

✅ data\genel_bilgiler.txt okundu (1476 karakter)
✅ data\kultur.txt okundu (2557 karakter)
✅ data\sehirler.txt okundu (2004 karakter)
✅ data\tarih_ekonomi.txt okundu (2672 karakter)
✅ data\turizm.txt okundu (2181 karakter)

📊 Toplam: 10898 karakter
📊 Toplam: 1572 kelime


Hücre 4: Chunking Yapalım

In [3]:
# Text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

print("✂️  Chunking yapılıyor...")

# Chunk'lara böl
chunks = text_splitter.split_text(full_knowledge)

print(f"✅ {len(chunks)} chunk oluşturuldu")
print(f"📏 Ortalama chunk boyutu: {sum(len(c) for c in chunks) / len(chunks):.0f} karakter")

# İlk 3 chunk'ı görelim
print("\n" + "=" * 70)
print("📚 İLK 3 CHUNK ÖRNEĞİ:")
print("=" * 70)

for i in range(min(3, len(chunks))):
    print(f"\n--- CHUNK {i+1} ({len(chunks[i])} karakter) ---")
    print(chunks[i])

✂️  Chunking yapılıyor...
✅ 28 chunk oluşturuldu
📏 Ortalama chunk boyutu: 405 karakter

📚 İLK 3 CHUNK ÖRNEĞİ:

--- CHUNK 1 (267 karakter) ---
TÜRKİYE HAKKINDA GENEL BİLGİLER

Başkent ve Resmi Bilgiler:
Türkiye'nin başkenti Ankara'dır. 1923 yılında Mustafa Kemal Atatürk tarafından başkent ilan edilmiştir.
Resmi dili Türkçe'dir. Para birimi Türk Lirası (TL)'dır.
Türkiye'nin resmi adı Türkiye Cumhuriyeti'dir.

--- CHUNK 2 (283 karakter) ---
Coğrafi Bilgiler:
Türkiye'nin toplam yüzölçümü 783,562 km²'dir.
3 tarafı denizlerle çevrilidir: Karadeniz, Akdeniz ve Ege Denizi.
Avrupa ve Asya kıtalarını birbirine bağlar.
İstanbul, hem Avrupa hem Asya kıtasında yer alan tek şehirdir.
Boğazlar: İstanbul Boğazı ve Çanakkale Boğazı.

--- CHUNK 3 (433 karakter) ---
Nüfus Bilgileri:
Türkiye'nin nüfusu yaklaşık 85 milyon civarındadır (2024).
En kalabalık şehir İstanbul'dur (yaklaşık 16 milyon).
İkinci en kalabalık şehir Ankara'dır (yaklaşık 5.7 milyon).
Üçüncü en kalabalık şehir İzmir'dir (yaklaşık 4.4 mi

Hücre 5: ChromaDB Hazırlayalım

In [4]:
print("🧠 Embedding modeli yükleniyor...")

# Embedding fonksiyonu
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# ChromaDB client
client = chromadb.Client()

# Eski collection'ı sil (varsa)
try:
    client.delete_collection("turkiye_bilgi")
except:
    pass

# Yeni collection
collection = client.create_collection(
    name="turkiye_bilgi",
    embedding_function=sentence_transformer_ef
)

print("✅ Vector Database hazır!")

🧠 Embedding modeli yükleniyor...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1095.75it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector Database hazır!


Hücre 6: Chunk'ları Vector DB'ye Ekleyelim

In [5]:
print(f"💾 {len(chunks)} chunk vector database'e ekleniyor...")
print("⏳ Bu işlem 30-60 saniye sürebilir...")

# Chunk'ları ekle
collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    metadatas=[{"chunk_id": i} for i in range(len(chunks))]
)

print(f"✅ {len(chunks)} chunk başarıyla eklendi!")
print("🎉 Türkiye Bilgi Bankası hazır!")

💾 28 chunk vector database'e ekleniyor...
⏳ Bu işlem 30-60 saniye sürebilir...
✅ 28 chunk başarıyla eklendi!
🎉 Türkiye Bilgi Bankası hazır!


Hücre 7: Test Araması Yapalım

In [6]:
# Test soruları
test_sorular = [
    "Türkiye'nin başkenti neresi?",
    "En kalabalık şehir hangisi?",
    "Kapadokya nerede?",
]

print("=" * 70)
print("🧪 TEST ARAMALARI")
print("=" * 70)

for soru in test_sorular:
    print(f"\n🔍 Soru: {soru}")
    print("📚 İlgili bilgiler aranıyor...\n")
    
    results = collection.query(
        query_texts=[soru],
        n_results=2
    )
    
    for i, (doc, dist) in enumerate(zip(results['documents'][0], results['distances'][0])):
        print(f"🏆 Sonuç {i+1} - Benzerlik: {1-dist:.3f}")
        print("-" * 70)
        print(doc[:200] + "..." if len(doc) > 200 else doc)
        print()

🧪 TEST ARAMALARI

🔍 Soru: Türkiye'nin başkenti neresi?
📚 İlgili bilgiler aranıyor...

🏆 Sonuç 1 - Benzerlik: 0.709
----------------------------------------------------------------------
TÜRKİYE HAKKINDA GENEL BİLGİLER

Başkent ve Resmi Bilgiler:
Türkiye'nin başkenti Ankara'dır. 1923 yılında Mustafa Kemal Atatürk tarafından başkent ilan edilmiştir.
Resmi dili Türkçe'dir. Para birimi T...

🏆 Sonuç 2 - Benzerlik: 0.672
----------------------------------------------------------------------
Nüfus Bilgileri:
Türkiye'nin nüfusu yaklaşık 85 milyon civarındadır (2024).
En kalabalık şehir İstanbul'dur (yaklaşık 16 milyon).
İkinci en kalabalık şehir Ankara'dır (yaklaşık 5.7 milyon).
Üçüncü en ...


🔍 Soru: En kalabalık şehir hangisi?
📚 İlgili bilgiler aranıyor...

🏆 Sonuç 1 - Benzerlik: 0.633
----------------------------------------------------------------------
TÜRKİYE TURİZM BİLGİLERİ

En Çok Turist Alan Şehirler:
1. İstanbul - Yıllık 15+ milyon turist, tarihi mekanlar, alışveriş
2. Antalya - 1

Hücre 8: RAG + Ollama Chatbot Fonksiyonu

In [7]:
def turkiye_chatbot(soru, n_results=3):
    """
    Türkiye hakkında soru sor, RAG + Ollama ile cevap al
    """
    print("=" * 70)
    print(f"🇹🇷 TÜRKIYE BİLGİ CHATBOT'U")
    print("=" * 70)
    print(f"💬 Soru: {soru}\n")

    # 1. RETRIEVAL - Vector DB'de ara (documents + distances alınıyor)
    print("📚 İlgili bilgiler aranıyor...")

    results = collection.query(
        query_texts=[soru],
        n_results=n_results,
        include=["documents", "distances"]
    )

    retrieved_docs = results['documents'][0]
    distances = results.get('distances', [[]])[0]

    if not retrieved_docs:
        return "❌ İlgili bilgi bulunamadı."

    # Basit similarity hesapla (Chroma distance -> similarity = 1 - distance)
    similarities = [1 - d for d in distances] if distances else [0.0 for _ in retrieved_docs]
    best_sim = max(similarities) if similarities else 0.0
    SIMILARITY_THRESHOLD = 0.60  # ayarlanabilir eşik (deneme)

    # İlgililik log'u
    print("\n📊 Retrieval sonuçları (önizleme ve skorlar):")
    for i, (doc, dist, sim) in enumerate(zip(retrieved_docs, distances, similarities)):
        print(f"[{i+1}] similarity={sim:.3f} distance={dist:.3f} preview={doc[:150]}...")

    # Eğer en iyi sonuç eşik altında ise LLM'i çağırma
    if best_sim < SIMILARITY_THRESHOLD:
        print(f"❌ En yakın parça yeterince ilgili değil (benzerlik={best_sim:.3f} < {SIMILARITY_THRESHOLD}). LLM çağrılmıyor.")
        return "Bu soru dokümanlarımda yok; ek kaynak lazım."

    print(f"✅ {len(retrieved_docs)} ilgili bilgi bulundu (en iyi benzerlik={best_sim:.3f})\n")

    # 2. AUGMENTATION - Prompt hazırla (sadece Türkçe, kaynak belirtilmesi isteniyor)
    context = "\n\n---\n\n".join(retrieved_docs)

    prompt = f"""Sen Türkiye hakkında bilgi veren yardımcı bir asistansın.

KAYNAK BİLGİLER:
{context}

KULLANICI SORUSU: {soru}

ÖNEMLİ:
- Yalnızca sağlanan CONTEXT'e dayanarak cevap ver.
- CONTEXT'te yoksa kesinlikle "Bu bilgi kaynaklarımda yok" de ve tahmin etme.
- Cevabı sadece Türkçe yaz; başka dil veya karakter kullanma.
- Cevabında hangi kaynak parçasını kullandığını numara ile belirt (örn. [1], [2]).

CEVAP:"""

    # 3. GENERATION - Ollama ile cevap (deterministik, daha güvenli)
    print("🤖 Ollama cevap hazırlıyor...\n")

    try:
        # İlk deneme: temperature/top_p argümanları ile (yeni veya beklenen API)
        try:
            response = ollama.generate(
                model='llama3.2',
                prompt=prompt,
                temperature=0,
                top_p=0.8
            )
        except TypeError:
            # Farklı ollama istemcisi sürümleri temperature/top_p kabul etmeyebilir
            try:
                response = ollama.generate(model='llama3.2', prompt=prompt)
            except Exception as inner_e:
                raise inner_e

        # response farklı şekillerde gelebilir; güvenli şekilde string'e çevir
        answer = ''
        try:
            if isinstance(response, dict):
                answer = response.get('response') or response.get('content') or str(response)
            else:
                answer = getattr(response, 'response', None) or getattr(response, 'content', None) or str(response)
        except Exception:
            answer = str(response)

        print("=" * 70)
        print("💬 CEVAP:")
        print("=" * 70)
        print(answer)

        # (FUZZY KALDIRILDI) Kaynakları göster ve cevabı döndür
        print("\n" + "=" * 70)
        print("📚 KAYNAK BİLGİLER:")
        print("=" * 70)
        for i, doc in enumerate(retrieved_docs):
            print(f"\n[{i+1}] {doc[:150]}...")

        return answer

    except Exception as e:
        error_msg = f"❌ Hata: {str(e)}\n\n💡 Ollama çalışıyor mu kontrol edin!"
        print(error_msg)
        return error_msg

Daha Sıkı Kontrol RAG

In [8]:
def turkiye_chatbot_strict(soru, n_results=3, similarity_threshold=0.7):
    """
    Daha sıkı kontrol - halüsinasyonu önle
    """
    print("=" * 70)
    print(f"🇹🇷 TÜRKIYE BİLGİ CHATBOT'U (Strict Mode)")
    print("=" * 70)
    print(f"💬 Soru: {soru}\n")
    
    # 1. RETRIEVAL
    results = collection.query(
        query_texts=[soru],
        n_results=n_results
    )
    
    retrieved_docs = results['documents'][0]
    distances = results['distances'][0]
    
    # ⚠️ YENİ: Benzerlik skorunu kontrol et
    similarities = [1 - d for d in distances]
    
    print("📊 Benzerlik skorları:")
    for i, sim in enumerate(similarities):
        print(f"   Chunk {i+1}: {sim:.3f}")
    
    # Eğer hiçbir chunk yeterince benzer değilse
    if max(similarities) < similarity_threshold:
        print(f"\n⚠️  En yüksek benzerlik: {max(similarities):.3f}")
        print(f"⚠️  Eşik değeri: {similarity_threshold}")
        print("\n❌ CEVAP:")
        print("=" * 70)
        answer = "Üzgünüm, bu konuda bilgi bankamda yeterli bilgi yok. Lütfen Türkiye'nin genel özellikleri, şehirleri, turizm, kültür, tarih veya ekonomi hakkında soru sorun."
        print(answer)
        return answer
    
    print(f"\n✅ Yeterli benzerlik var, cevap hazırlanıyor...\n")
    
    # 2. AUGMENTATION - Daha sıkı prompt
    context = "\n\n---\n\n".join(retrieved_docs)
    
    prompt = f"""Sen Türkiye hakkında bilgi veren bir asistandsın.

KAYNAK BİLGİLER:
{context}

KULLANICI SORUSU: {soru}

ÇOK ÖNEMLİ KURALLAR:
1. SADECE ve SADECE yukarıdaki kaynak bilgilerde olan şeyleri söyle
2. Eğer kaynaklarda cevap YOKSA: "Bu bilgi kaynaklarımda yok" de
3. KENDİ BİLGİNİ EKLEME! Sadece kaynakları kullan
4. Emin değilsen cevap verme
5. Türkçe cevap ver

CEVAP:"""
    
    # 3. GENERATION
    print("🤖 Ollama cevap hazırlıyor...\n")
    
    try:
        response = ollama.generate(
            model='llama3.2',
            prompt=prompt,
            options={
                'temperature': 0.1,  # ⚠️ YENİ: Daha deterministik
            }
        )
        
        answer = response['response']
        
        print("=" * 70)
        print("💬 CEVAP:")
        print("=" * 70)
        print(answer)
        
        return answer
        
    except Exception as e:
        return f"❌ Hata: {str(e)}"

print("✅ Strict mode fonksiyonu hazır!")

✅ Strict mode fonksiyonu hazır!


Hücre 9: Test Edelim!

In [9]:
# TEST SORULARI

sorular = [
    "Türkiye'nin başkenti neresi?",
    "En kalabalık şehir hangisi?",
    "İstanbul'da hangi turistik yerler var?",
    "Türk kahvesi nedir?",
    "Kapadokya'da ne yapılır?",
    "Türkiye kaç komşuya sahip?",
    "Gaziantep neyle ünlüdür?",
    "Türkiye'nin milli marşı nedir?",
]

for soru in sorular:
    turkiye_chatbot(soru, n_results=3)
    print("\n" + "⏸️ " * 35 + "\n")

🇹🇷 TÜRKIYE BİLGİ CHATBOT'U
💬 Soru: Türkiye'nin başkenti neresi?

📚 İlgili bilgiler aranıyor...

📊 Retrieval sonuçları (önizleme ve skorlar):
[1] similarity=0.709 distance=0.291 preview=TÜRKİYE HAKKINDA GENEL BİLGİLER

Başkent ve Resmi Bilgiler:
Türkiye'nin başkenti Ankara'dır. 1923 yılında Mustafa Kemal Atatürk tarafından başkent ila...
[2] similarity=0.672 distance=0.328 preview=Nüfus Bilgileri:
Türkiye'nin nüfusu yaklaşık 85 milyon civarındadır (2024).
En kalabalık şehir İstanbul'dur (yaklaşık 16 milyon).
İkinci en kalabalık ...
[3] similarity=0.671 distance=0.329 preview=TÜRKİYE TURİZM BİLGİLERİ

En Çok Turist Alan Şehirler:
1. İstanbul - Yıllık 15+ milyon turist, tarihi mekanlar, alışveriş
2. Antalya - 14+ milyon turi...
✅ 3 ilgili bilgi bulundu (en iyi benzerlik=0.709)

🤖 Ollama cevap hazırlıyor...

💬 CEVAP:
Turkey'nin başkenti Ankara'dır.

📚 KAYNAK BİLGİLER:

[1] TÜRKİYE HAKKINDA GENEL BİLGİLER

Başkent ve Resmi Bilgiler:
Türkiye'nin başkenti Ankara'dır. 1923 yılında Mustafa Kema

In [13]:
# KÖTÜ ÖRNEK (halüsinasyon yapıyordu)
turkiye_chatbot_strict("Türkiye'nin en kalabalık şehri hangisidir??", similarity_threshold=0.6)

print("\n" + "=" * 70 + "\n")

# İYİ ÖRNEK (TXT'te var)
turkiye_chatbot_strict("Türkiye'nin başkenti neresi?", similarity_threshold=0.6)

🇹🇷 TÜRKIYE BİLGİ CHATBOT'U (Strict Mode)
💬 Soru: Türkiye'nin en kalabalık şehri hangisidir??

📊 Benzerlik skorları:
   Chunk 1: 0.752
   Chunk 2: 0.751
   Chunk 3: 0.741

✅ Yeterli benzerlik var, cevap hazırlanıyor...

🤖 Ollama cevap hazırlıyor...

💬 CEVAP:
Türkiye'nin en kalabalık şehri İstanbul'dur.


🇹🇷 TÜRKIYE BİLGİ CHATBOT'U (Strict Mode)
💬 Soru: Türkiye'nin başkenti neresi?

📊 Benzerlik skorları:
   Chunk 1: 0.709
   Chunk 2: 0.672
   Chunk 3: 0.671

✅ Yeterli benzerlik var, cevap hazırlanıyor...

🤖 Ollama cevap hazırlıyor...

💬 CEVAP:
Türkiye'nin başkenti Ankara'dır.


"Türkiye'nin başkenti Ankara'dır."

Sıkı Kontrol Test 

Hücre 10: İnteraktif Chatbot

In [11]:
def interaktif_chatbot():
    """
    Kullanıcı ile interaktif sohbet
    """
    print("=" * 70)
    print("🇹🇷 TÜRKİYE BİLGİ CHATBOT'U")
    print("=" * 70)
    print("""
Merhaba! Ben Türkiye hakkında bilgi veren bir chatbot'um.
Bana Türkiye ile ilgili sorular sorabilirsiniz!

Örnek sorular:
  • Türkiye'nin başkenti neresi?
  • En kalabalık şehir hangisi?
  • İstanbul'da gezilecek yerler neler?
  • Türk mutfağı nedir?
  • Kapadokya hakkında bilgi ver
  • Atatürk kim?

Komutlar:
  'çıkış' - Sohbeti bitir
  'örnekler' - Örnek soruları göster
""")
    print("=" * 70)
    
    # Örnek sorular listesi
    ornek_sorular = [
        "Türkiye'nin başkenti neresi?",
        "En kalabalık şehir hangisi?",
        "Kapadokya nerede?",
        "İstanbul'da ne gezilir?",
        "Türk kahvesi nedir?",
        "Adana kebap nedir?",
        "Türkiye kaç ile sahip?",
        "Pamukkale nerede?",
        "Efes antik kenti nerede?",
        "Van Gölü nerede?",
    ]
    
    while True:
        print("\n💬 Sorunuzu yazın:")
        soru = input(">>> ").strip()
        
        # Komut kontrolü
        if soru.lower() in ['çıkış', 'cikis', 'exit', 'quit']:
            print("\n👋 Görüşmek üzere! İyi günler!")
            break
        
        if soru.lower() in ['örnekler', 'ornekler', 'examples']:
            print("\n📝 ÖRNEK SORULAR:")
            print("=" * 70)
            for i, ornek in enumerate(ornek_sorular, 1):
                print(f"{i}. {ornek}")
            continue
        
        if not soru:
            print("❌ Lütfen bir soru yazın!")
            continue
        
        # RAG ile cevap
        print("\n" + "⏳" * 35)
        turkiye_chatbot(soru, n_results=3)

print("✅ İnteraktif chatbot hazır!")
print("Başlatmak için sonraki hücreyi çalıştırın.")

✅ İnteraktif chatbot hazır!
Başlatmak için sonraki hücreyi çalıştırın.


Hücre 11: Sistemi Başlat!

In [12]:
interaktif_chatbot()

🇹🇷 TÜRKİYE BİLGİ CHATBOT'U

Merhaba! Ben Türkiye hakkında bilgi veren bir chatbot'um.
Bana Türkiye ile ilgili sorular sorabilirsiniz!

Örnek sorular:
  • Türkiye'nin başkenti neresi?
  • En kalabalık şehir hangisi?
  • İstanbul'da gezilecek yerler neler?
  • Türk mutfağı nedir?
  • Kapadokya hakkında bilgi ver
  • Atatürk kim?

Komutlar:
  'çıkış' - Sohbeti bitir
  'örnekler' - Örnek soruları göster


💬 Sorunuzu yazın:

👋 Görüşmek üzere! İyi günler!
